# Searching a shared collection with CLIP embeddings

The first notebook showed how IIIF delivers images at the size and region you ask for. The second showed how Europeana collects descriptions of objects from thousands of institutions. This notebook uses both ideas for a different kind of search: finding images by **what they look like**, not by what their metadata says.

The approach rests on an **embedding**: a list of numbers (here 512) that a neural network computes from an image, so that images with similar content get similar numbers. [CLIP](https://openai.com/research/clip) was trained on images paired with captions, and it computes embeddings for both images and short texts *in the same space*. A sentence such as "a ship on a stormy sea" can therefore be compared directly with the embedding of every image in a collection.

In this notebook you will:

1. Load a **shared dataset** of small images from many Europeana institutions, together with their precomputed CLIP embeddings.
2. Embed your own **text query** and rank the whole collection by similarity.
3. Restrict the search by institution, country or period, and see which institutions the results come from.
4. Search with an **image** instead of a text.
5. **Look closely** at a result by requesting a larger version or a detail from its institution, through IIIF where available.

## 1. How the dataset was made

The dataset contains **no images**, only numbers and links. It was prepared once by the instructor, on a computing cluster:

```
Europeana Search API   →   metadata records (title, creator, institution, rights, links)
        │
        ▼   for every record, while it is processed:
   request a small version of the image
        ├─ the media URL is a IIIF image service  →  from that service (448 px)
        └─ otherwise                              →  from Europeana's thumbnail service (400 px wide)
        │
        ▼
   CLIP image encoder   →   one vector of 512 numbers per image   (the image itself is then discarded)

stored: the vectors, and for each image the URL of its small version
```

Only openly reusable images were included. A small version is all CLIP needs: it resizes every image to 224 × 224 pixels anyway. The **original, full-size files stay at the institutions**, and so do the small versions: when a search returns results, your computer fetches *those* images from their URLs, a dozen at a time, and nothing else. This is the IIIF idea from the first notebook applied to a whole collection.

The dataset folder contains:

| Path | Content |
|---|---|
| `metadata.csv` | one row per image: title, creator, year, institution, country, rights, the URL of the small version (`small_image_url`), the original file (`full_image_url`), and the IIIF image service when one answered |
| `embeddings/clip_vit_b32.npz` | the CLIP vectors, in the same order as `metadata.csv` |
| `models/` | the CLIP model weights |

## Setup

In [ ]:
import csv
import hashlib
import html
import os
from base64 import b64encode
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from io import BytesIO
from pathlib import Path

import numpy as np
import requests
from IPython.display import HTML, display
from PIL import Image

NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent

# The shared dataset. Set the environment variable DSI_WORKSHOP_DIR to use a different location.
DATA = Path(os.environ.get("DSI_WORKSHOP_DIR", PROJECT_ROOT / "data" / "DSI-Workshop"))
CACHE_DIR = PROJECT_ROOT / "data" / "image_cache"      # downloaded small images are kept here
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Dataset folder:", DATA)

## 2. Load the collection

`metadata.csv` and the embeddings are stored in the same order, which is checked here. Row *i* of the metadata describes vector *i* of the embeddings. Everything is loaded into memory except the images.

In [ ]:
with open(DATA / "metadata.csv", encoding="utf-8") as f:
    meta = list(csv.DictReader(f))

npz = np.load(DATA / "embeddings" / "clip_vit_b32.npz")
V = npz["vectors"]                                   # (N, 512), already L2-normalised
assert list(npz["ids"]) == [r["id"] for r in meta], "metadata.csv and embeddings are not aligned"

institution = np.array([r["institution"] for r in meta])
country     = np.array([r["country"] for r in meta])
year        = np.array([int(r["year"]) if r["year"].isdigit() else 0 for r in meta])
has_iiif    = np.array([bool(r["iiif_service"]) for r in meta])

print(f"{len(meta):,} images from {len(set(institution)):,} institutions in {len(set(country))} countries")
print(f"embeddings: {V.shape}, {V.dtype}, {V.nbytes / 1e6:,.1f} MB")
print(f"images with a working IIIF image service: {has_iiif.sum():,} ({has_iiif.mean():.1%})")
print(f"images with a year: {(year > 0).sum():,}")

In [ ]:
def load_small(i):
    # The small image of row i: from the local cache if present, otherwise from its URL. Returns (PIL image, bytes).
    path = CACHE_DIR / (hashlib.md5(meta[i]["id"].encode()).hexdigest() + ".jpg")
    if not path.exists():
        r = requests.get(meta[i]["small_image_url"], timeout=45)
        r.raise_for_status()
        path.write_bytes(r.content)
    return Image.open(path).convert("RGB"), path.stat().st_size

def small_image(i):
    try:
        return load_small(i)[0]
    except Exception:
        return Image.new("RGB", (170, 170), "#ddd")         # the source did not answer

def thumb(i, size=170):
    # An <img> tag with the image embedded, so that it shows in any notebook front end.
    im = small_image(i)
    im.thumbnail((size * 2, size * 2))
    buf = BytesIO(); im.save(buf, "JPEG", quality=80)
    return (f"<img src='data:image/jpeg;base64,{b64encode(buf.getvalue()).decode()}' "
            f"style='width:{size}px;height:{size}px;object-fit:cover;background:#eee'>")

def show_grid(indices, scores=None, size=170):
    with ThreadPoolExecutor(8) as ex:                          # fetch the images of the results in parallel
        list(ex.map(small_image, indices))
    cards = ""
    for i in indices:
        r = meta[i]
        score = f"<br><b>similarity {scores[i]:.3f}</b>" if scores is not None else ""
        yr = f", {r['year']}" if r["year"] else ""
        cards += (f"<figure style='margin:0;width:{size}px;font:11px sans-serif'>{thumb(i, size)}"
                  f"<figcaption><b>{html.escape(r['title'][:60])}</b><br>{html.escape(r['institution'][:38])}{yr}{score}</figcaption></figure>")
    display(HTML(f"<div style='display:flex;gap:12px;flex-wrap:wrap'>{cards}</div>"))

rng = np.random.default_rng(1)
print("A random sample of the collection (the images are fetched from their institutions now):")
show_grid(rng.choice(len(meta), 12, replace=False))

The collection is deliberately mixed: photographs, prints, paintings, documents, natural history specimens, objects. Which institutions contribute the most images?

In [ ]:
def bar_table(pairs, total, title):
    rows = "".join(f"<tr><td style='padding-right:10px'>{html.escape(k[:48])}</td><td style='text-align:right;padding-right:8px'>{v:,}</td>"
                   f"<td><div style='background:#2a78d6;height:10px;width:{max(v / total * 220, 1):.0f}px'></div></td></tr>" for k, v in pairs)
    return f"<div><b>{title}</b><table style='font:12px sans-serif'>{rows}</table></div>"

by_inst = Counter(institution).most_common(10)
by_country = Counter(country).most_common(10)
display(HTML("<div style='display:flex;gap:40px;flex-wrap:wrap'>"
             + bar_table(by_inst, by_inst[0][1], "Institutions with the most images")
             + bar_table(by_country, by_country[0][1], "Countries with the most images") + "</div>"))

## 3. Load CLIP

The model has two halves: an **image encoder** (already applied to the whole collection) and a **text encoder**. For a text query only the text encoder runs, which is fast on an ordinary laptop CPU. The weights are loaded from the dataset folder.

In [ ]:
import torch
import clip

device = "cuda" if torch.cuda.is_available() else "cpu"
model_dirs = [DATA / "models", DATA / "models" / "CLIP", PROJECT_ROOT / "models" / "CLIP"]
model_dir = next((d for d in model_dirs if (d / "ViT-B-32.pt").exists()), model_dirs[0])

model, preprocess = clip.load("ViT-B/32", device=device, download_root=str(model_dir))
model.eval()
print(f"CLIP ViT-B/32 loaded on {device} from {model_dir}")

## 4. Search with text

Encoding a query gives a vector in the same 512-dimensional space as the images. Both are normalised to length 1, so the **dot product** of two vectors equals their cosine similarity: 1 means the same direction, values near 0 mean unrelated. Ranking the collection is one matrix–vector product.

In [ ]:
def encode_text(text):
    with torch.no_grad():
        f = model.encode_text(clip.tokenize([text], truncate=True).to(device)).float()
        f = f / f.norm(dim=-1, keepdim=True)
    return f[0].cpu().numpy()

def similarity(q, chunk=200_000):
    # Dot product of the query with every image vector (in chunks, so large collections fit in memory).
    out = np.empty(len(V), dtype=np.float32)
    for s in range(0, len(V), chunk):
        out[s:s + chunk] = V[s:s + chunk].astype(np.float32) @ q
    return out

def top_k(scores, k=12, mask=None):
    s = scores if mask is None else np.where(mask, scores, -np.inf)
    idx = np.argpartition(-s, min(k, len(s) - 1))[:k]
    idx = idx[np.argsort(-s[idx])]
    return idx[np.isfinite(s[idx])]

In [ ]:
# ============================================================
# EXERCISE: change the query and run the cell
# ============================================================
QUERY = "a ship on a stormy sea"      # <-- try "a portrait of a woman in a red dress", "a map of a city", "gold coins"
# ============================================================

scores = similarity(encode_text(QUERY))
best = top_k(scores, 12)
print(f"Top results for: {QUERY!r}\n")
show_grid(best, scores)

Tips for queries:

| Instead of | Try |
|---|---|
| `water` | `a river flowing through a landscape` |
| `old` | `a faded black and white photograph` |
| `art` | `an oil painting of flowers in a vase` |

CLIP understands descriptions better than single keywords, and it works best in English. It also matches *style and medium* ("a pencil drawing", "a photograph") as well as content, which is useful for a mixed collection.

### Restricting the search

Because the metadata is aligned with the vectors, a filter is just a boolean mask applied before ranking. For example, only images dated before 1900, or only from certain countries:

In [ ]:
QUERY = "a portrait of a person"
mask = (year > 0) & (year < 1900)                       # dated, and before 1900
# mask = np.isin(country, ["Sweden", "Finland", "Norway"])   # alternative: by country
# mask = np.isin(institution, ["Wellcome Collection"])       # alternative: by institution

scores = similarity(encode_text(QUERY))
print(f"{mask.sum():,} of {len(mask):,} images pass the filter. Top results for {QUERY!r}:\n")
show_grid(top_k(scores, 12, mask), scores)

## 5. Where do the results come from?

A search result is only as varied as the collection it searches. Comparing the institutions in the top results with the institutions in the whole collection shows whether a query favours some sources, and which institutions hold the kind of image you asked for.

In [ ]:
QUERY = "a ship on a stormy sea"
scores = similarity(encode_text(QUERY))
top = top_k(scores, 100)

in_top = Counter(institution[top]).most_common(8)
share_all = Counter(institution)
rows = "".join(
    f"<tr><td style='padding-right:10px'>{html.escape(name[:48])}</td><td style='text-align:right;padding-right:10px'>{n}%</td>"
    f"<td style='text-align:right;padding-right:10px'>{share_all[name] / len(meta):.1%}</td></tr>" for name, n in in_top)
display(HTML(f"<b>{html.escape(QUERY)}</b>: institutions in the top 100 results<table style='font:12px sans-serif'>"
             "<tr><th style='text-align:left'>Institution</th><th>share of top 100</th><th>share of collection</th></tr>" + rows + "</table>"))

## 6. Search with an image

The same space holds image embeddings, so an image can be the query. Two options: take an image from the collection ("more like this"), or use your own image file.

In [ ]:
def encode_image(pil_image):
    with torch.no_grad():
        f = model.encode_image(preprocess(pil_image.convert("RGB")).unsqueeze(0).to(device)).float()
        f = f / f.norm(dim=-1, keepdim=True)
    return f[0].cpu().numpy()

YOUR_IMAGE = None            # <-- e.g. "my_photo.jpg" (a file next to the notebook), or None to use a collection image
SEED_INDEX = int(top[0])     # the best match from the ship query above

if YOUR_IMAGE:
    query_image = Image.open(YOUR_IMAGE)
    q = encode_image(query_image)
    exclude = None
    print("Query: your image")
else:
    query_image = small_image(SEED_INDEX)
    q = V[SEED_INDEX].astype(np.float32)
    exclude = SEED_INDEX
    print("Query: collection image", SEED_INDEX, "-", meta[SEED_INDEX]["title"][:60])

display(query_image.copy().resize((200, int(200 * query_image.height / query_image.width))))

scores = similarity(q)
if exclude is not None:
    scores[exclude] = -1                      # do not return the query itself
show_grid(top_k(scores, 12), scores)

## 7. Look closely: the larger version, on request

The small versions are enough for CLIP but not for studying a detail. Once a result is interesting, a larger version can be requested from the institution. Two cases:

- **The record has a working IIIF image service** (the `iiif_service` column). One URL returns a larger version, or just a region of the image, and nothing else is downloaded.
- **There is no image service.** The only option is the original file (`full_image_url`), which has to be downloaded completely, and is often several megabytes. Any zoom or crop is then done locally.

The function below does both, and reports how much data was transferred:

In [ ]:
def look_closely(i, size=1000, region=None):
    # region: (x, y, w, h) as fractions of the image, e.g. (0.25, 0.25, 0.5, 0.5) for the central quarter.
    r = meta[i]
    small, small_bytes = load_small(i)

    if r["iiif_service"]:
        reg = "full" if region is None else "pct:" + ",".join(f"{v * 100:.2f}" for v in region)
        url = f"{r['iiif_service']}/{reg}/!{size},{size}/0/default.jpg"
        data = requests.get(url, timeout=60).content
        big, how = Image.open(BytesIO(data)).convert("RGB"), "IIIF image service"
    else:
        url = r["full_image_url"]
        data = requests.get(url, timeout=120).content
        big, how = Image.open(BytesIO(data)).convert("RGB"), "original file (no image service)"
        if region is not None:
            W, H = big.size
            big = big.crop((int(region[0] * W), int(region[1] * H), int((region[0] + region[2]) * W), int((region[1] + region[3]) * H)))
        big.thumbnail((size, size))

    print(f"{r['title'][:70]}\n{r['institution']}")
    print(f"request:  {how}\n          {url}")
    print(f"small version used for the embedding: {small.size[0]} x {small.size[1]}, {small_bytes / 1024:,.0f} KB")
    print(f"requested version:             {big.size[0]} x {big.size[1]}, transferred {len(data) / 1024:,.0f} KB")
    if r["landing_page"]:
        print("object page:", r["landing_page"])

    h = min(520, big.height)
    strip = Image.new("RGB", (int(small.width * h / small.height) + int(big.width * h / big.height) + 12, h), "white")
    a = small.resize((int(small.width * h / small.height), h)); b = big.resize((int(big.width * h / big.height), h))
    strip.paste(a, (0, 0)); strip.paste(b, (a.width + 12, 0))
    display(strip)

First a result that has an image service. The search is restricted to those with `has_iiif`, and the request asks for a detail (the central quarter of the image) at 1,000 pixels:

In [ ]:
QUERY = "a ship on a stormy sea"
scores = similarity(encode_text(QUERY))
iiif_hits = top_k(scores, 6, has_iiif)

if len(iiif_hits):
    show_grid(iiif_hits, scores)
    look_closely(int(iiif_hits[0]), size=1000, region=(0.25, 0.25, 0.5, 0.5))
else:
    print("No image in this collection has a working IIIF image service.")

Then the best match overall, which may have no image service. Compare the amount of data transferred for the larger version:

In [ ]:
overall = int(top_k(scores, 1)[0])
look_closely(overall, size=1000)

## Summary

- An **embedding** turns an image or a text into a vector. CLIP puts both in one space, so a sentence can be compared with images and an image with images.
- Searching a precomputed collection is a **matrix–vector product**, plus optional masks built from the metadata.
- The collection is a sample that reflects what institutions made available. Results show *what is in the collection*, so it pays to check where they come from.
- Small images are enough to compute embeddings. A larger version or a detail is requested **only when needed**, through the IIIF image API where the institution offers one, and as a full download where not.

### Limits to keep in mind

- CLIP was trained on images and captions from the web. It reflects the language, subjects and biases of that data, and works best on English queries and on photographic or pictorial content.
- Similarity is not meaning. Two images can be close because of composition, colour or medium and not because of subject.
- The dataset is a selection (openly reusable images, up to a fixed number per institution), not Europeana as a whole.
- The images live at the institutions. If a source is offline or has moved the file, that image shows as a grey square, and the links in `metadata.csv` reflect the time the dataset was built.
- Detecting IIIF services from the media URL finds many but not all of them. Some institutions have a service that cannot be seen from the URL, and are served through Europeana's thumbnails.